In [ ]:
!pip install -q osmnx powerlaw python-igraph
!pip install osmnx

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = "/content/drive/MyDrive/proyecto-redes-bogota"
for sub in ["data/raw", "data/processed", "figures"]:
    os.makedirs(f"{BASE}/{sub}", exist_ok=True)

In [ ]:
import osmnx as ox
import networkx as nx
print(ox.__version__)   # el código asume 2.x

In [ ]:
import osmnx as ox, networkx as nx, geopandas as gpd, random
random.seed(42)

bog = ox.geocode_to_gdf("Bogotá, Colombia").to_crs(9377)
sumapaz = ox.geocode_to_gdf("Sumapaz, Bogotá, Colombia").to_crs(9377)
urbano = bog.geometry.iloc[0].difference(sumapaz.geometry.iloc[0])
poly = gpd.GeoSeries([urbano], crs=9377).to_crs(4326).iloc[0]

In [ ]:
filtro = '["highway"~"^(motorway|trunk|primary|secondary)(_link)?$"]'

def construir(custom_filter=None):
    G = ox.graph_from_polygon(poly, network_type="drive", custom_filter=custom_filter)
    G = ox.routing.add_edge_speeds(G)
    G = ox.routing.add_edge_travel_times(G)
    n0 = len(G)
    G = ox.truncate.largest_component(G, strongly=True)
    print(f"  nodos {n0} -> {len(G)} ({len(G)/n0:.1%}), aristas {G.number_of_edges()}")
    return ox.projection.project_graph(G, to_crs="EPSG:9377")

print("Arterial:"); G_art = construir(filtro)
ox.io.save_graphml(G_art, f"{BASE}/data/processed/gvial_arterial.graphml")

print("Drive completo:"); G_drive = construir()
ox.io.save_graphml(G_drive, f"{BASE}/data/processed/gvial_drive.graphml")

In [ ]:
D = ox.convert.to_digraph(G_art, weight="travel_time")   # dirigido, sin paralelas
U = D.to_undirected()

N, M = D.number_of_nodes(), D.number_of_edges()
print(f"N={N}, M={M}, grado medio={2*U.number_of_edges()/N:.2f}")
print(f"Clustering medio={nx.average_clustering(U):.4f}")

# Longitud media de camino en minutos (fuentes muestreadas)
fuentes = random.sample(list(D.nodes), min(500, N))
tiempos = [t for s in fuentes
           for v, t in nx.single_source_dijkstra_path_length(D, s, weight="travel_time").items()
           if v != s]
print(f"Tiempo medio de viaje (flujo libre): {sum(tiempos)/len(tiempos)/60:.1f} min")

In [ ]:
eb = nx.edge_betweenness_centrality(D, k=1000, weight="travel_time", seed=42)
nx.set_edge_attributes(D, eb, "eb")

top = sorted(eb.items(), key=lambda x: x[1], reverse=True)[:20]
for i, ((u, v), b) in enumerate(top, 1):
    nombre = D.edges[u, v].get("name", "sin nombre")
    lat, lon = G_art.nodes[u]["lat"] if "lat" in G_art.nodes[u] else None, None
    print(f"{i:2d}. {nombre} | eb={b:.4f}")

In [ ]:
for u, v, k, data in G_art.edges(keys=True, data=True):
    data["eb"] = eb.get((u, v), 0)

ec = ox.plot.get_edge_colors_by_attr(G_art, "eb", cmap="inferno")
fig, ax = ox.plot.plot_graph(G_art, node_size=0, edge_color=ec,
                             edge_linewidth=1, bgcolor="white", show=False)
fig.savefig(f"{BASE}/figures/betweenness_arterial.png", dpi=200, bbox_inches="tight")

In [ ]:
import pandas as pd, numpy as np

rows = []
for (u, v), b in eb.items():
    d = D.edges[u, v]
    nombre = d.get("name", "sin nombre")
    if isinstance(nombre, list):
        nombre = " / ".join(sorted(set(nombre)))
    hw = d.get("highway")
    rows.append({"u": u, "v": v, "nombre": nombre,
                 "highway": hw if isinstance(hw, str) else str(hw),
                 "length_m": d["length"], "speed_kph": d.get("speed_kph"), "eb": b})
df = pd.DataFrame(rows)

corredores = (df.groupby("nombre")
                .agg(eb_max=("eb", "max"), eb_medio=("eb", "mean"),
                     km=("length_m", lambda s: s.sum() / 1000), tramos=("eb", "size"))
                .sort_values("eb_max", ascending=False))
corredores.head(15)

In [ ]:
x = np.sort(df["eb"].values)              # ascendente
n = len(x)
gini = (2 * np.sum(np.arange(1, n + 1) * x) / (n * x.sum())) - (n + 1) / n
top1 = x[-max(1, n // 100):].sum() / x.sum()
print(f"Gini de edge betweenness: {gini:.3f}")
print(f"El 1 % de tramos concentra el {top1:.1%} de la intermediación total")

In [ ]:
df.groupby("highway")["speed_kph"].describe()[["count", "mean", "min", "max"]]

In [ ]:
eb_len = nx.edge_betweenness_centrality(D, k=1000, weight="length", seed=42)
df["eb_len"] = df.apply(lambda r: eb_len[(r.u, r.v)], axis=1)

top_t = set(df.groupby("nombre")["eb"].max().nlargest(10).index)
top_l = set(df.groupby("nombre")["eb_len"].max().nlargest(10).index)
print("Coinciden:", top_t & top_l)
print(f"Jaccard top 10 corredores: {len(top_t & top_l) / len(top_t | top_l):.2f}")

In [ ]:
for u, v, k, data in G_art.edges(keys=True, data=True):
    data["eb_log"] = np.log10(eb.get((u, v), 0) + 1e-5)

ec = ox.plot.get_edge_colors_by_attr(G_art, "eb_log", cmap="inferno")
fig, ax = ox.plot.plot_graph(G_art, node_size=0, edge_color=ec, edge_linewidth=1.2,
                             bgcolor="white", show=False, close=False)
fig.savefig(f"{BASE}/figures/betweenness_arterial_log.png", dpi=250, bbox_inches="tight")

In [ ]:
import time
D_full = ox.convert.to_digraph(G_drive, weight="travel_time")
t0 = time.time()
eb_full = nx.edge_betweenness_centrality(D_full, k=50, weight="travel_time", seed=42)
dt = time.time() - t0
print(f"k=50: {dt:.0f} s  →  estimado k=500: {dt*10/60:.0f} min")

In [ ]:
eb_t = nx.edge_betweenness_centrality(D, weight="travel_time")   # exacta, sin k
eb_l = nx.edge_betweenness_centrality(D, weight="length")
df["eb"] = df.apply(lambda r: eb_t[(r.u, r.v)], axis=1)
df["eb_len"] = df.apply(lambda r: eb_l[(r.u, r.v)], axis=1)

In [ ]:
from scipy.stats import spearmanr

nom = df[df["nombre"] != "sin nombre"].copy()
nom["eb_x_len"] = nom["eb"] * nom["length_m"]
nom["ebl_x_len"] = nom["eb_len"] * nom["length_m"]

corr = nom.groupby("nombre").agg(km=("length_m", lambda s: s.sum() / 1000),
                                 eb_max=("eb", "max"),
                                 carga_t=("eb_x_len", "sum"),
                                 carga_l=("ebl_x_len", "sum"))
corr["carga_t"] /= corr["km"] * 1000      # eb medio ponderado por longitud
corr["carga_l"] /= corr["km"] * 1000
corr = corr[corr["km"] >= 2]              # excluye fragmentos

top_t = set(corr["carga_t"].nlargest(10).index)
top_l = set(corr["carga_l"].nlargest(10).index)
print(f"Jaccard corredores: {len(top_t & top_l) / len(top_t | top_l):.2f}")
print(f"Spearman tramos (todos): {spearmanr(df['eb'], df['eb_len'])[0]:.2f}")
corr.sort_values("carga_t", ascending=False).head(10)

In [ ]:
from shapely.geometry import Point
import geopandas as gpd

sn = df[df["nombre"] == "sin nombre"].nlargest(8, "eb")
pts = gpd.GeoSeries([Point(D.nodes[u]["x"], D.nodes[u]["y"]) for u in sn["u"]],
                    crs=9377).to_crs(4326)
for (_, r), p in zip(sn.iterrows(), pts):
    print(f"{r.highway:15s} eb={r.eb:.3f}  https://www.google.com/maps?q={p.y:.6f},{p.x:.6f}")

In [ ]:
t0 = time.time()
eb_full = nx.edge_betweenness_centrality(D_full, k=1000, weight="travel_time", seed=42)
print(f"{(time.time() - t0) / 60:.1f} min")
import pickle
pickle.dump(eb_full, open(f"{BASE}/data/processed/eb_drive_k1000.pkl", "wb"))

In [ ]:
corr["dep_vel"] = corr["carga_t"] / corr["carga_l"]
corr["tipo"] = pd.cut(
    corr["dep_vel"],
    [0, 1.5, 3, np.inf],
    labels=["estructural", "mixto", "por velocidad"]
)

relevantes = corr.loc[list(top_t | top_l)].sort_values("carga_t", ascending=False)
relevantes[["km", "carga_t", "carga_l", "dep_vel", "tipo"]]

In [ ]:
def tabla_tramos(D, eb):
    rows = []
    for (u, v), b in eb.items():
        d = D.edges[u, v]
        nombre = d.get("name", "sin nombre")
        if isinstance(nombre, list):
            nombre = " / ".join(sorted(set(nombre)))
        rows.append({"u": u, "v": v, "nombre": nombre, "length_m": d["length"], "eb": b})
    return pd.DataFrame(rows)

def corredores(df, min_km=2):
    t = df[df["nombre"] != "sin nombre"].copy()
    t["eb_x_len"] = t["eb"] * t["length_m"]
    c = t.groupby("nombre").agg(km=("length_m", lambda s: s.sum() / 1000),
                                eb_max=("eb", "max"), carga=("eb_x_len", "sum"))
    c["carga"] /= c["km"] * 1000
    return c[c["km"] >= min_km]

corr_drive = corredores(tabla_tramos(D_full, eb_full))
corr_drive.sort_values("carga", ascending=False).head(15)

In [ ]:
comunes = corr.index.intersection(corr_drive.index)
rho = spearmanr(corr.loc[comunes, "carga_t"], corr_drive.loc[comunes, "carga"])[0]

top_art = set(corr["carga_t"].nlargest(10).index)
top_drv = set(corr_drive["carga"].nlargest(10).index)
print(f"Corredores comunes: {len(comunes)} | Spearman carga: {rho:.2f}")
print("Top 10 en ambos:", top_art & top_drv)
print("Solo en drive:", top_drv - top_art)

In [ ]:
from collections import Counter

sur = ["Avenida Calle 22 Sur", "Avenida Calle 51 Sur", "Avenida Calle 57R Sur",
       "Avenida Calle 61 Sur", "Avenida Calle 68 Sur", "Avenida Carrera 19C"]

def nombre_str(n):
    return " / ".join(sorted(set(n))) if isinstance(n, list) else n

cnt = Counter()
for u, v, d in D_full.edges(data=True):
    nm = nombre_str(d.get("name"))
    if nm in sur:
        hw = d.get("highway")
        cnt[(nm, hw if isinstance(hw, str) else str(hw))] += 1
for k, v in sorted(cnt.items()):
    print(k, v)
print("Presentes en G_arterial:", [s for s in sur if s in set(df["nombre"])])

In [ ]:
filtro_v2 = '["highway"~"^(motorway|trunk|primary|secondary|tertiary)(_link)?$"]'

In [ ]:
rng = random.Random(42)
FUENTES = rng.sample(list(D.nodes), 300)

def eficiencia(G, fuentes=FUENTES):
    n = G.number_of_nodes()
    tot = 0.0
    for s in fuentes:
        dist = nx.single_source_dijkstra_path_length(G, s, weight="travel_time")
        tot += sum(1 / t for v, t in dist.items() if v != s and t > 0)
    return tot / (len(fuentes) * (n - 1))

def con_reverso(G, edges):
    """Cerrar un tramo cierra ambos sentidos."""
    s = set(edges)
    s |= {(v, u) for u, v in edges if G.has_edge(v, u)}
    return list(s)

E0 = eficiencia(D)
print(f"E0 = {E0:.6f}")

In [ ]:
def cerrar_corredor(nombre):
    tramos = df[df["nombre"].str.contains(nombre, regex=False)]
    H = D.copy()
    H.remove_edges_from(con_reverso(D, list(zip(tramos.u, tramos.v))))
    return H, tramos["length_m"].sum() / 1000

def cerrar_aleatorio(km_obj, seed):
    r = random.Random(seed)
    edges = list(D.edges(data="length"))
    r.shuffle(edges)
    quitar, km = [], 0
    for u, v, L in edges:
        if km >= km_obj: break
        quitar.append((u, v)); km += L / 1000
    H = D.copy(); H.remove_edges_from(con_reverso(D, quitar))
    return H

candidatos = ["Avenida Carrera 30", "Avenida Carrera 72", "Avenida Carrera 68",
              "Avenida Carrera 86", "Avenida Carrera 45", "Avenida Calle 26",
              "Transversal 35", "Avenida Calle 45A Sur"]
res = []
for nom in candidatos:
    H, km = cerrar_corredor(nom)
    caida = 100 * (1 - eficiencia(H) / E0)
    control = np.mean([100 * (1 - eficiencia(cerrar_aleatorio(km, s)) / E0) for s in range(5)])
    res.append({"corredor": nom, "km": round(km, 1), "caida_E_%": caida,
                "control_aleatorio_%": control, "veces_control": caida / control})
res_corr = pd.DataFrame(res).sort_values("caida_E_%", ascending=False)
res_corr.to_csv(f"{BASE}/data/processed/cierre_corredores.csv", index=False)
res_corr

In [ ]:
def curva(estrategia, pasos=15, frac=0.01, seed=0):
    H = D.copy()
    lote = int(frac * D.number_of_edges())
    r = random.Random(seed)
    orden_ini = sorted(eb_t, key=eb_t.get, reverse=True)
    out = [(0.0, 1.0, 1.0)]
    for p in range(1, pasos + 1):
        if estrategia == "aleatorio":
            quitar = r.sample(list(H.edges()), lote)
        elif estrategia == "eb_inicial":
            quitar = [e for e in orden_ini if H.has_edge(*e)][:lote]
        else:  # eb_recalculada
            ebr = nx.edge_betweenness_centrality(H, k=300, weight="travel_time", seed=p)
            quitar = sorted(ebr, key=ebr.get, reverse=True)[:lote]
        H.remove_edges_from(con_reverso(H, quitar))
        S = len(max(nx.strongly_connected_components(H), key=len)) / H.number_of_nodes()
        out.append((p * frac, S, eficiencia(H) / E0))
        print(f"{estrategia} paso {p}: S={S:.3f}, E={out[-1][2]:.3f}")
    return pd.DataFrame(out, columns=["f", "S", "E_rel"]).assign(estrategia=estrategia, rep=seed)

curvas = pd.concat([curva("aleatorio", seed=s) for s in range(5)] +
                   [curva("eb_inicial"), curva("eb_recalculada")])
curvas.to_csv(f"{BASE}/data/processed/curvas_resiliencia.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for metrica, ax in zip(["S", "E_rel"], axes):
    for est, g in curvas.groupby("estrategia"):
        m = g.groupby("f")[metrica].agg(["mean", "std"])
        ax.plot(m.index, m["mean"], marker="o", ms=3, label=est)
        if est == "aleatorio":
            ax.fill_between(m.index, m["mean"] - m["std"], m["mean"] + m["std"], alpha=0.2)
    ax.set_xlabel("Fracción de tramos retirados (f)")
    ax.set_ylabel({"S": "Componente gigante S(f)", "E_rel": "Eficiencia E(f)/E(0)"}[metrica])
    ax.legend()
fig.tight_layout()
fig.savefig(f"{BASE}/figures/resiliencia.png", dpi=200)

In [ ]:
def ataque_recalculado_log(pasos=7, frac=0.01):
    H = D.copy()
    lote = int(frac * D.number_of_edges())
    log = []
    for p in range(1, pasos + 1):
        ebr = nx.edge_betweenness_centrality(H, k=300, weight="travel_time", seed=p)
        quitar = con_reverso(H, sorted(ebr, key=ebr.get, reverse=True)[:lote])
        H.remove_edges_from(quitar)
        log += [(p, u, v) for u, v in quitar]
        comps = sorted(nx.strongly_connected_components(H), key=len, reverse=True)
        print(f"paso {p}: tamaños de las 4 mayores componentes {[len(c) for c in comps[:4]]}")
    return H, pd.DataFrame(log, columns=["paso", "u", "v"])

H7, log = ataque_recalculado_log()
log.to_csv(f"{BASE}/data/processed/log_ataque_recalculado.csv", index=False)

corte = log[log["paso"] >= 6].merge(df[["u", "v", "nombre"]], on=["u", "v"], how="left")
corte["nombre"].value_counts().head(15)

In [ ]:
comps = sorted(nx.strongly_connected_components(H7), key=len, reverse=True)[:4]
paleta = ["#1f77b4", "#ff7f0e", "#2ca02c", "#9467bd"]
color = {n: paleta[i] for i, c in enumerate(comps) for n in c}
nc = [color.get(n, "#cccccc") for n in G_art.nodes]

crit = set(zip(corte["u"], corte["v"]))
ec = ["red" if (u, v) in crit else "#dddddd" for u, v, k in G_art.edges(keys=True)]
ew = [2.5 if (u, v) in crit else 0.4 for u, v, k in G_art.edges(keys=True)]

fig, ax = ox.plot.plot_graph(G_art, node_color=nc, node_size=4, edge_color=ec,
                             edge_linewidth=ew, bgcolor="white", show=False, close=False)
fig.savefig(f"{BASE}/figures/corte_critico.png", dpi=250, bbox_inches="tight")

In [ ]:
for ax in axes:
    ax.axvline(0.065, color="green", ls="--", lw=1)
    ax.axvline(0.14, color="tab:blue", ls="--", lw=1)

In [ ]:
# ¿El corte es físico o efecto de los sentidos viales?
print("Débiles:", sorted(map(len, nx.weakly_connected_components(H7)), reverse=True)[:5])

# ¿Qué se retiró en cada paso? (confirmar que la NQS cayó antes del colapso)
pasos = log.merge(df[["u", "v", "nombre"]], on=["u", "v"], how="left")
for p, g in pasos.groupby("paso"):
    print(p, g["nombre"].value_counts().head(4).to_dict())

In [ ]:
def agregar_enlace(G, u, v, vel_kph=50, factor=1.2, nombre="CORREDOR NUEVO"):
    H = G.copy()
    L = factor * np.hypot(G.nodes[u]["x"] - G.nodes[v]["x"], G.nodes[u]["y"] - G.nodes[v]["y"])
    t = L / (vel_kph / 3.6)
    for a, b in [(u, v), (v, u)]:
        H.add_edge(a, b, length=L, travel_time=t, name=nombre)
    return H, L / 1000

def acelerar(G, nombres, vel_kph=50):
    H = G.copy()
    for u, v, d in H.edges(data=True):
        if nombre_str(d.get("name")) in nombres:
            d["travel_time"] = d["length"] / (vel_kph / 3.6)
    return H

def robustez(G, pasos=10, frac=0.01):
    H = G.copy()
    lote = int(frac * G.number_of_edges())
    S = [1.0]
    for p in range(1, pasos + 1):
        ebr = nx.edge_betweenness_centrality(H, k=300, weight="travel_time", seed=p)
        H.remove_edges_from(con_reverso(H, sorted(ebr, key=ebr.get, reverse=True)[:lote]))
        S.append(len(max(nx.strongly_connected_components(H), key=len)) / H.number_of_nodes())
    return np.mean(S), S

R0, S0 = robustez(D)
print(f"Robustez base R0 = {R0:.3f}")

In [ ]:
def candidatos_costura(zona_a, zona_b, n=150, dmin=800, dmax=4000, top=8):
    r = random.Random(1)
    sa, sb = r.sample(list(zona_a), n), r.sample(list(zona_b), n)
    xy = lambda x: np.array([D.nodes[x]["x"], D.nodes[x]["y"]])
    filas = []
    for u in sa:
        dist = nx.single_source_dijkstra_path_length(D, u, weight="length")
        for v in sb:
            de = np.linalg.norm(xy(u) - xy(v))
            if dmin <= de <= dmax and v in dist:
                filas.append((u, v, de, dist[v] / de))
    c = pd.DataFrame(filas, columns=["u", "v", "d_euc", "rodeo"]).sort_values("rodeo", ascending=False)
    elegidos = []                       # evita candidatos casi idénticos (>1 km entre sí)
    for row in c.itertuples():
        if all(np.linalg.norm(xy(row.u) - xy(e.u)) > 1000 for e in elegidos):
            elegidos.append(row)
        if len(elegidos) == top:
            break
    return pd.DataFrame(elegidos)

cand = candidatos_costura(comps[0], comps[2])   # azul (957) y verde (718)

res = []
for r in cand.itertuples():
    H, km = agregar_enlace(D, r.u, r.v)
    dE = 100 * (eficiencia(H) / E0 - 1)
    res.append({"u": r.u, "v": r.v, "km": km, "rodeo": r.rodeo, "dE_%": dE, "dE_por_km": dE / km})
res_cand = pd.DataFrame(res).sort_values("dE_por_km", ascending=False)
res_cand

In [ ]:
# Velocidad actual de los corredores subutilizados (para justificar el escenario B)
print(df[df["nombre"].isin(["Avenida Carrera 68", "Avenida Carrera 50"])]
        .groupby("nombre")["speed_kph"].mean())

escenarios = {}
for i, r in enumerate(res_cand.head(3).itertuples(), 1):
    H, km = agregar_enlace(D, r.u, r.v)
    escenarios[f"A{i} enlace {km:.1f} km"] = H
escenarios["B Cra 68 y Cra 50 a 50 km/h"] = acelerar(D, {"Avenida Carrera 68", "Avenida Carrera 50"})

filas = []
for nom, H in escenarios.items():
    R, S = robustez(H)
    filas.append({"escenario": nom, "R": R, "dR_%": 100 * (R / R0 - 1),
                  "dE_%": 100 * (eficiencia(H) / E0 - 1)})
    print(nom, f"R={R:.3f}")
res_esc = pd.DataFrame(filas)
res_esc.to_csv(f"{BASE}/data/processed/escenarios_robustez.csv", index=False)
res_esc

In [ ]:
def tiempo_entre_zonas(G, za, zb, n=100):
    r = random.Random(3)
    sa, sb = r.sample(list(za), n), set(r.sample(list(zb), n))
    ts = []
    for u in sa:
        d = nx.single_source_dijkstra_path_length(G, u, weight="travel_time")
        ts += [d[v] for v in sb if v in d]
    return np.mean(ts) / 60

base = tiempo_entre_zonas(D, comps[0], comps[2])
print(f"Base azul→verde: {base:.1f} min")
for nom, H in escenarios.items():
    t = tiempo_entre_zonas(H, comps[0], comps[2])
    print(f"{nom}: {t:.1f} min ({100 * (t / base - 1):+.1f} %)")

In [ ]:
eb_base = nx.edge_betweenness_centrality(D, k=1000, weight="travel_time", seed=42)
eb_B = nx.edge_betweenness_centrality(escenarios["B Cra 68 y Cra 50 a 50 km/h"],
                                      k=1000, weight="travel_time", seed=42)
c0 = corredores(tabla_tramos(D, eb_base))
cB = corredores(tabla_tramos(D, eb_B))
comp = pd.DataFrame({"base": c0["carga"], "B": cB["carga"]})
comp["cambio_%"] = 100 * (comp["B"] / comp["base"] - 1)
comp.loc[["Avenida Carrera 30", "Avenida Carrera 68", "Avenida Carrera 50",
          "Avenida Carrera 72", "Transversal 35"]]

In [ ]:
def robustez(G, pasos=10, frac=0.01, offset=0):
    H = G.copy()
    lote = int(frac * G.number_of_edges())
    S = [1.0]
    for p in range(1, pasos + 1):
        ebr = nx.edge_betweenness_centrality(H, k=300, weight="travel_time", seed=p + offset)
        H.remove_edges_from(con_reverso(H, sorted(ebr, key=ebr.get, reverse=True)[:lote]))
        S.append(len(max(nx.strongly_connected_components(H), key=len)) / H.number_of_nodes())
    return np.mean(S), S

H_AB, _ = agregar_enlace(escenarios["B Cra 68 y Cra 50 a 50 km/h"], 261630424, 261040900)
casos = {"base": D,
         "A3": escenarios["A3 enlace 4.2 km"],
         "B": escenarios["B Cra 68 y Cra 50 a 50 km/h"],
         "A3+B": H_AB}

R_prev = {"base": R0, "A3": 0.7295, "B": 0.6986}     # corrida ya hecha (offset 0)
filas = []
for nom, H in casos.items():
    reps = ([R_prev[nom]] if nom in R_prev else [robustez(H)[0]])
    reps += [robustez(H, offset=o)[0] for o in (100, 200)]
    filas.append({"escenario": nom, "R_media": np.mean(reps), "R_std": np.std(reps),
                  "dE_%": 100 * (eficiencia(H) / E0 - 1)})
    print(nom, [round(r, 3) for r in reps])
res_final = pd.DataFrame(filas)
res_final["dR_%"] = 100 * (res_final["R_media"] / res_final.loc[0, "R_media"] - 1)
res_final.to_csv(f"{BASE}/data/processed/escenarios_final.csv", index=False)
res_final

In [ ]:
pts = gpd.GeoSeries([Point(D.nodes[n]["x"], D.nodes[n]["y"]) for n in (261630424, 261040900)],
                    crs=9377).to_crs(4326)
for n, p in zip((261630424, 261040900), pts):
    print(n, f"https://www.google.com/maps?q={p.y:.6f},{p.x:.6f}")

In [ ]:
import matplotlib.pyplot as plt

CIAN, NARANJA, LILA = "#22D3EE", "#F97316", "#A78BFA"
FONDO, TEXTO, GRIS = "#0B1220", "#E5E7EB", "#6B7280"

plt.rcParams.update({
    "figure.facecolor": FONDO, "axes.facecolor": FONDO, "savefig.facecolor": FONDO,
    "axes.edgecolor": GRIS, "axes.labelcolor": TEXTO, "text.color": TEXTO,
    "xtick.color": TEXTO, "ytick.color": TEXTO, "font.size": 13,
    "axes.spines.top": False, "axes.spines.right": False,
})

def guardar(fig, nombre):
    fig.savefig(f"{BASE}/figures/{nombre}.png", dpi=250, bbox_inches="tight")

In [ ]:
grados = pd.Series(dict(U.degree()))
pct = 100 * grados.value_counts().sort_index() / len(grados)
etq = lambda v: "<1 %" if v < 1 else f"{v:.0f} %"

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.bar(pct.index, pct.values, color=CIAN, width=0.7)
for k, v in pct.items():
    ax.text(k, v + 1.5, etq(v), ha="center")
ax.axvline(grados.mean(), color=NARANJA, ls="--", lw=1.5)
ax.text(grados.mean() + 0.45, pct.max() * 0.55, f"media {grados.mean():.1f}".replace(".", ","),
        color=NARANJA)
ax.set_ylim(0, pct.max() * 1.15)
ax.set_xticks(pct.index)
ax.set_xlabel("Grado (vías que llegan a la intersección)")
ax.set_ylabel("% de intersecciones")
guardar(fig, "fig_grado")

In [ ]:
from matplotlib.lines import Line2D

def grupo(r):
    if r >= 2:   return "Atraen por velocidad (>2×)"
    if r <= 0.6: return "Subutilizados (<0,6×)"
    return "Estructurales (0,6–2×)"

COLORES = {"Atraen por velocidad (>2×)": NARANJA,
           "Estructurales (0,6–2×)": CIAN,
           "Subutilizados (<0,6×)": LILA}

sel = corr.loc[corr["carga_t"].nlargest(12).index.union(corr["carga_l"].nlargest(12).index)].copy()
sel["grupo"] = sel["dep_vel"].apply(grupo)

fig, ax = plt.subplots(figsize=(8, 6))
lim = max(sel["carga_t"].max(), sel["carga_l"].max()) * 1.1
xs = np.array([0, lim])

# Bandas: el color de cada punto coincide con la zona donde cae
ax.fill_between(xs, 2 * xs, lim * 3, color=NARANJA, alpha=0.08, lw=0)
ax.fill_between(xs, 0, 0.6 * xs, color=LILA, alpha=0.10, lw=0)
ax.plot(xs, 2 * xs, color=NARANJA, lw=0.8, alpha=0.6)
ax.plot(xs, 0.6 * xs, color=LILA, lw=0.8, alpha=0.6)
ax.plot(xs, xs, color=GRIS, ls="--", lw=1)

for g, d in sel.groupby("grupo"):
    ax.scatter(d["carga_l"], d["carga_t"], s=40 + d["km"] * 3,
               color=COLORES[g], alpha=0.9, edgecolor="none")

claves = sel.loc[sel.index.isin(NOMBRES)]
textos = [ax.text(r.carga_l, r.carga_t, NOMBRES[n], fontsize=11) for n, r in claves.iterrows()]
adjust_text(textos, ax=ax, arrowprops=dict(arrowstyle="-", color=GRIS, lw=0.5))

handles = [Line2D([0], [0], marker="o", ls="", color=c, markersize=9, label=g)
           for g, c in COLORES.items()]
ax.legend(handles=handles, frameon=False, loc="upper left")
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.set_xlabel("Carga medida por distancia")
ax.set_ylabel("Carga medida por tiempo")
guardar(fig, "fig_tipos_corredor")

In [ ]:
H_A3 = escenarios["A3 enlace 4.2 km"]
eb_A3 = nx.edge_betweenness_centrality(H_A3, k=1000, weight="travel_time", seed=42)
cA3 = corredores(tabla_tramos(H_A3, eb_A3))
nqs_A3 = 100 * (cA3.loc["Avenida Carrera 30", "carga"] / c0.loc["Avenida Carrera 30", "carga"] - 1)
nqs_B = comp.loc["Avenida Carrera 30", "cambio_%"]
print(f"Carga NQS: A3 {nqs_A3:+.1f} %, B {nqs_B:+.1f} %")

# Robustez: diferencias pareadas por semilla (réplicas de la celda AG)
R = {"base": [0.668, 0.697, 0.669], "A3": [0.730, 0.696, 0.705], "B": [0.699, 0.692, 0.728]}
dR = {k: 100 * (np.array(R[k]) - np.array(R["base"])) / np.array(R["base"]) for k in ("A3", "B")}

metricas = ["Carga NQS\n(↓ mejor)", "Tiempo centro-occ. → sur\n(↓ mejor)", "Robustez R*\n(↑ mejor)"]
vals = {"A. Enlace 4,2 km": [nqs_A3, -0.8, dR["A3"].mean()],
        "B. Cra. 68 y 50 a 50 km/h": [nqs_B, -3.8, dR["B"].mean()]}
errs = {"A. Enlace 4,2 km": [0, 0, dR["A3"].std()],
        "B. Cra. 68 y 50 a 50 km/h": [0, 0, dR["B"].std()]}

x = np.arange(len(metricas)); w = 0.36
fig, ax = plt.subplots(figsize=(9, 5))
for i, (nom, color) in enumerate(zip(vals, [GRIS, CIAN])):
    barras = ax.bar(x + (i - 0.5) * w, vals[nom], w, yerr=errs[nom], color=color,
                    label=nom, capsize=4, error_kw={"ecolor": TEXTO, "lw": 1})
    for b, v, e in zip(barras, vals[nom], errs[nom]):
        y = v + e + 0.6 if v >= 0 else v - 1.8
        ax.text(b.get_x() + b.get_width() / 2, y, f"{v:+.1f} %".replace(".", ","),
                ha="center", fontsize=11)
ax.axvline(0, color=GRIS, lw=1)
ax.set_xticks(x); ax.set_xticklabels(metricas)
ax.set_ylabel("Cambio frente a la red actual (%)")
ax.legend(frameon=False)
fig.text(0.01, -0.02, "*Media de 3 réplicas pareadas; barras de error = desviación estándar. No concluyente.",
         fontsize=9, color=GRIS)
ax.set_ylim(-18, 11)
guardar(fig, "fig_escenarios")

In [ ]:
for ax in axes:
    ax.axvline(0.065, color="#22C55E", ls="--", lw=1.2)
    ax.axvline(0.14, color=CIAN, ls="--", lw=1.2)
axes[0].text(0.067, 0.95, "f_c adaptativo\n6,5 %", fontsize=10, color="#22C55E", va="top")
axes[0].text(0.142, 0.95, "f_c azar\n14 %", fontsize=10, color=CIAN, va="top")